# 6-Level vSTIRAP Simulation (Adiabatically Eliminated Model)

This notebook simulates coherent spin-photon entanglement generation via virtual Stimulated Raman Adiabatic Passage (vSTIRAP).
It uses an adiabatically eliminated 6-level effective Hamiltonian to remove fast optical oscillations and vastly speed up the ODE solver, leveraging a **'Coupling Ket'** outer-product approach for clean generation of AC Stark shifts and Raman couplings.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qutip import basis, mesolve, expect

# 1. Hilbert Space Definition (6-dimensional)
# basis(6, 0) = |1> (Ground)
# basis(6, 1) = |2> (Ground)
# basis(6, 2) = |5> (Target)
# basis(6, 3) = |6> (Target)
# basis(6, 4) = |7> (Leakage)
# basis(6, 5) = |8> (Leakage)
dim = 6

In [ ]:
# 2. Physical Parameters (units: µeV and µeV^-1)
Delta_L1 = 1500.0
Delta_L_minus = 1510.0
g = 20.0
E_7 = 130.0
E_8 = -130.0

Omega_max = 150.0

eps_H = 0.10 * np.exp(1j * 0.0)
eps_V = 0.10 * np.exp(1j * np.pi / 4.0)

t0 = 2500.0
sigma = 650.0

def pulse_Omega(t, **kwargs):
    return Omega_max * np.exp(-((t - t0)**2) / (2 * sigma**2))

def pulse_Omega_sq(t, **kwargs):
    return pulse_Omega(t)**2

In [ ]:
# 3. The "Coupling Ket" Construction (Adiabatic Elimination)

# A. Couplings to the Virtual |T_+> state
ket_L_plus = (1 + eps_V)*basis(6, 0) + (1 + eps_H)*basis(6, 1)
ket_C_plus = g * (basis(6, 2) + basis(6, 3) + eps_H*basis(6, 4) + eps_V*basis(6, 5))

# B. Couplings to the Virtual |T_-> state
ket_L_minus = eps_H*basis(6, 0) + eps_V*basis(6, 1)
ket_C_minus = g * (eps_V*basis(6, 2) + eps_H*basis(6, 3))

In [ ]:
# 4. Building the Effective Hamiltonian Matrix
def proj(i, j):
    return basis(6, i) * basis(6, j).dag()

# 1. H_static (Time-Independent)
H0 = E_7 * proj(4, 4) + E_8 * proj(5, 5)
H_cav_cav = -(1/Delta_L1) * (ket_C_plus * ket_C_plus.dag()) - (1/Delta_L_minus) * (ket_C_minus * ket_C_minus.dag())
H_static = H0 + H_cav_cav

# 2. H_Stark (Scales with Omega^2(t))
H_Stark = -(1/Delta_L1) * (ket_L_plus * ket_L_plus.dag()) - (1/Delta_L_minus) * (ket_L_minus * ket_L_minus.dag())

# 3. H_Raman (Scales with Omega(t))
H_Raman_plus = -(1/Delta_L1) * (ket_C_plus * ket_L_plus.dag() + ket_L_plus * ket_C_plus.dag())
H_Raman_minus = -(1/Delta_L_minus) * (ket_C_minus * ket_L_minus.dag() + ket_L_minus * ket_C_minus.dag())
H_Raman = H_Raman_plus + H_Raman_minus

# Setup the time-dependent effective Hamiltonian list for mesolve
H_eff = [H_static, [H_Stark, pulse_Omega_sq], [H_Raman, pulse_Omega]]

In [ ]:
# 5. Initial State & Simulation Setup

# Initial State: Symmetric ground-state superposition
psi0 = (basis(6, 0) + basis(6, 1)).unit()
rho0 = psi0 * psi0.dag()

# Target Bell State for Fidelity
psi_target = (basis(6, 2) + basis(6, 3)).unit()

# Time Grid
tlist = np.linspace(0, 5000, 2000)

# Run mesolve with headroom for leakage state oscillations
result = mesolve(H_eff, rho0, tlist, c_ops=[], options={'nsteps': 500000})

In [ ]:
# Outputs
t_ns = tlist / 1519.26

# Calculate populations and fidelity
populations = {
    '1 (Ground)': [expect(proj(0, 0), rho).real for rho in result.states],
    '2 (Ground)': [expect(proj(1, 1), rho).real for rho in result.states],
    '5 (Target)': [expect(proj(2, 2), rho).real for rho in result.states],
    '6 (Target)': [expect(proj(3, 3), rho).real for rho in result.states],
    '7 (Leakage)': [expect(proj(4, 4), rho).real for rho in result.states],
    '8 (Leakage)': [expect(proj(5, 5), rho).real for rho in result.states],
}
fidelity = [expect(psi_target * psi_target.dag(), rho).real for rho in result.states]

# 1. Plot population dynamics
plt.figure(figsize=(10, 6))
for label, pop in populations.items():
    plt.plot(t_ns, pop, label=f'State |{label}>')
plt.xlabel('Time (ns)')
plt.ylabel('Population')
plt.title('Population Dynamics (6-level Effective Model)')
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5))
plt.grid(True)
plt.tight_layout()
plt.show()

# 2. Plot instantaneous fidelity
plt.figure(figsize=(10, 6))
plt.plot(t_ns, fidelity, label='Target Fidelity', color='black', linewidth=2)
plt.xlabel('Time (ns)')
plt.ylabel('Fidelity')
plt.title('Instantaneous Fidelity')
plt.legend()
plt.grid(True)
plt.show()

# 3. Print final density matrix and compare populations
rho_final = result.states[-1]
print("Final Effective Density Matrix:")
print(np.round(rho_final.full(), 4))

print("\nFinal Populations:")
target_pop = populations['5 (Target)'][-1] + populations['6 (Target)'][-1]
leakage_pop = populations['7 (Leakage)'][-1] + populations['8 (Leakage)'][-1]
print(f"Target States Population: {target_pop:.4f}")
print(f"Leakage States Population: {leakage_pop:.4f}")
for label, pop in populations.items():
    print(f"State |{label}>: {pop[-1]:.4f}")